# Capstone — local research-paper draft

This notebook mirrors the paper at `work/paper/index.html`. The paper is a local draft only: model results, action-queue exports, and deployment are pending. Do not publish or replace `submission/paper_url.txt` until the analysis has been run and reviewed.

> The paper uses public-safe, evidence-matched language. No client names, domains, URLs, raw queries, or private content belong in this notebook or the paper.

## 1. Question

**Lane:** Refresh / Content Opportunity Scoring. **Question:** Which existing pages should a content editor inspect first, using safe content and search signals? **Decision:** how to allocate limited editorial review time. **Unit:** one eligible page in the starter snapshot. **Output:** a rule-ranked human-review queue with reason codes; the model comparison is a planned secondary check against the starter's current trend proxy.

A wrong positive costs review time; a missed candidate could delay noticing stale or underperforming content. Neither the starter proxy nor this observational snapshot measures the effect of a refresh.

In [ ]:
research_question = "Which existing content pages should an editor review first using safe search and content signals?"
lane = "Refresh / Content Opportunity Scoring"
unit_of_analysis = "one eligible pseudonymized content page"
intended_decision = "which page to inspect first; the score does not trigger automatic edits"
wrong_positive_cost = "editor review time spent on a page that did not need action"
wrong_negative_cost = "a potentially useful review is delayed or missed"

print(f"Lane: {lane}")
print(f"Research question: {research_question}")
print(f"Unit of analysis: {unit_of_analysis}")
print(f"Decision supported: {intended_decision}")
print("This is decision support, not a causal claim about refresh impact.")

## 2. Data

**Current draft source:** the repository’s `data/raw/content_refresh_anonymized.csv` starter slice, documented as 30,000 rows × 44 columns, one row per pseudonymized page across 32 pseudonymized clients. Metrics are trailing-90-day snapshot aggregates; the local dictionary does not identify an exact export date.

The analysis filters to pages with positive `impressions_90d` and `content_age_days >= 90`, then deduplicates by `content_id`. IDs are used only for deduplication or client-grouped splits. The full Hugging Face warehouse is not claimed as analyzed in this draft: the March 2026 contract code exists but was not run and has no verified outputs.

Excluded from public reporting: client names, domains, URLs, private queries, titles, raw text, credentials, and row-level identifiers. The starter dataset is the only source reflected in the current model and baseline code.

In [ ]:
from pathlib import Path

data_path_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
existing_data_paths = [str(path) for path in data_path_candidates if path.is_file()]
data_manifest = {
    "source_file": "data/raw/content_refresh_anonymized.csv",
    "documented_rows": 30000,
    "documented_columns": 44,
    "documented_client_groups": 32,
    "metric_window": "trailing 90-day snapshot; exact export date not stated in local dictionary",
    "local_path_found_from_current_directory": bool(existing_data_paths),
    "local_path_candidates_found": existing_data_paths,
    "full_warehouse_results_verified": False,
}
for key, value in data_manifest.items():
    print(f"{key}: {value}")

## 3. Methodology

**Proxy label:** `decline_proxy = 1` when the starter’s `trend_direction == "down"`; this is a contemporaneous bucket derived from the snapshot’s impression trend, not an independently observed future outcome.

**Week 5 model features:** content age, days since last update, search-volume estimate, competition, CPC, word count, character count, content type, and intent. Week 5 excludes `trend_direction`, `trend_pct`, impressions/clicks/sessions, and identifiers. Numeric missing values are intended to be imputed from training data; categoricals are imputed and one-hot encoded.

**Baseline:** Week 4 stale-visible rule: `days_since_last_update >= 180` and `impressions_90d >= 300`, ranked by impressions. The comparison uses the same eligible pages, client-held-out rows, proxy, and ranking metrics.

**Validation plan:** deterministic 75/25 client-held-out split; W06 also compares a stratified random-page split to expose split sensitivity. W06 includes a deliberately invalid feature derived from the outcome window and excludes it from the honest result. The starter data is cross-sectional, so this is not time-forward deployment validation. No metric is reportable until these notebook cells are run and reviewed.

In [ ]:
model_feature_spec = {
    "numeric_features": [
        "content_age_days",
        "days_since_last_update",
        "search_volume",
        "competition",
        "cpc",
        "word_count",
        "char_count",
    ],
    "categorical_features": ["content_type", "main_intent"],
    "target": "decline_proxy = 1 when trend_direction == 'down'",
    "target_scope": "current-snapshot trend-derived proxy; not a future outcome",
    "baseline": "stale-visible: days_since_last_update >= 180 and impressions_90d >= 300; score = impressions_90d",
    "validation": "75/25 GroupShuffleSplit by client_id; deterministic seed selection; compare same holdout rows",
    "leakage_exclusions": [
        "trend_direction",
        "trend_pct",
        "decline_proxy",
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
    ],
    "metric_status": "not verified; notebooks have not been run",
}

for section, detail in model_feature_spec.items():
    print(f"{section}: {detail}")

## 4. Results (vs baseline)

Saved W05/W06 output tables report the same-split comparison below. The target is a same-snapshot trend-derived decline proxy, not a future outcome; interpret every metric as agreement with that proxy. The local notebook copies lack execution counters, so reconcile these retained outputs and archive the metric receipt before public release.

In [ ]:
saved_model_comparison = [
    {"split": "Random page split", "method": "Week-4 stale-visible rule", "test_pages": 7500, "test_clients": 31, "proxy_rate": 0.542000, "Precision@10": 0.60, "Precision@50": 0.58, "Average precision": 0.542225},
    {"split": "Random page split", "method": "Week-5 Logistic Regression", "test_pages": 7500, "test_clients": 31, "proxy_rate": 0.542000, "Precision@10": 0.50, "Precision@50": 0.54, "Average precision": 0.639705},
    {"split": "Client-held-out", "method": "Week-4 stale-visible rule", "test_pages": 7115, "test_clients": 8, "proxy_rate": 0.516514, "Precision@10": 0.80, "Precision@50": 0.70, "Average precision": 0.517041},
    {"split": "Client-held-out", "method": "Week-5 Logistic Regression", "test_pages": 7115, "test_clients": 8, "proxy_rate": 0.516514, "Precision@10": 0.80, "Precision@50": 0.64, "Average precision": 0.528778},
]
print("Metrics transcribed from saved W05/W06 output tables; verify against a clean run before publication.")
for result in saved_model_comparison:
    print(result)

print("Client-held-out reading: rule leads by 0.06 at Precision@50; model average precision is higher by 0.011737.")
print("Random-to-grouped model average-precision difference: 0.110927; this is split sensitivity, not proof of memorization.")

saved_audit_findings = [
    {"signal": "Freshness vs trend", "verdict": "OPPOSITE", "evidence": "181+ day down share 47.1% (n=174) vs 0-90 day 51.2% (n=20,655)"},
    {"signal": "Position vs weighted CTR", "verdict": "CONFIRMED", "evidence": "Top-3 0.489% (n=555) vs deep 0.039% (n=878), pages with >=100 impressions"},
    {"signal": "Search volume vs impressions", "verdict": "MIXED", "evidence": "Median log impressions 6.736 (n=664) for 1000+ vs 6.774 (n=13,402) for 1-99"},
    {"signal": "Stale-visible coverage", "verdict": "OPPOSITE", "evidence": "22 of 174 stale pages meet the >=300-impression floor (12.6%)"},
]
print("Saved W04 signal-audit outputs:")
for finding in saved_audit_findings:
    print(finding)

saved_leakage_summary = {
    "clean_grouped_average_precision": 0.528778,
    "invalid_trend_pct_leakage_average_precision": 0.893436,
    "interpretation": "The higher score is invalid because trend_pct is derived from the label source.",
}
print("Saved W06 leakage-audit outputs:")
print(saved_leakage_summary)

## 5. Limitations

The starter target is a trend-derived proxy from the same snapshot; it does not measure future decline or refresh impact. The dataset is cross-sectional and limited to an eligible starter subset, and the retained notebook output blocks do not include local execution counts or locally archived result receipts. The paper cannot claim that a refresh causes recovery, that the model predicts Google, or that the starter-slice result generalizes to the full warehouse. On the grouped split, Logistic Regression has slightly higher average precision but lower Precision@50 than the transparent rule. The Week 7 priority scores are rule outputs, not probabilities or expected business value.

In [ ]:
claims_not_supported = [
    "A refresh causes a recovery in impressions or clicks.",
    "The model predicts Google's algorithm or future rankings.",
    "The starter-slice result generalizes to the full 79M-row warehouse.",
    "The rule score is a probability, expected traffic lift, or realized revenue.",
    "Logistic Regression is categorically better than the baseline: it has lower grouped Precision@50 despite slightly higher grouped average precision.",
    "A 0.893 average-precision score from trend_pct is valid: it is an intentional label-source leakage challenge.",
]
print("Claims this paper must not make:")
for claim in claims_not_supported:
    print(f"- {claim}")
print("Evidence-aligned framing: observed proxy comparisons and directional human decision support, not causal effects or future prediction.")
print("Provenance caveat: retained output blocks exist; local notebook execution counts and aggregate receipt files are absent.")

## 6. Ranked recommendations

The raw Week-7 rule queue assigns stale-visible pages its first tier. The signal audit found that staleness alone was opposite to the trend proxy, so the evidence-aware review order below does not blindly preserve that tier order. These remain human-review actions, not predicted lifts.

In [ ]:
ranked_action_mapping = [
    {
        "evidence_order": 1,
        "reason_code": "PAGE_ONE_CTR_REVIEW",
        "candidate_pages_in_saved_queue": 5456,
        "signal_evidence": "Position/CTR audit confirmed: top-3 weighted CTR 0.489% vs deep 0.039% on pages with >=100 impressions.",
        "human_action": "HUMAN_SNIPPET_REVIEW",
        "guardrail": "Pilot a capacity-limited batch; verify query mix, snippet relevance, and the reporting window.",
    },
    {
        "evidence_order": 2,
        "reason_code": "STRIKING_DISTANCE_REVIEW",
        "candidate_pages_in_saved_queue": 5043,
        "signal_evidence": "Rule identifies visible pages at positions 11-20; improvement is not validated by this comparison.",
        "human_action": "HUMAN_RELEVANCE_REVIEW",
        "guardrail": "Review intent and internal links; measure outcomes later, with no automatic expansion.",
    },
    {
        "evidence_order": 3,
        "reason_code": "STALE_VISIBLE",
        "candidate_pages_in_saved_queue": 22,
        "signal_evidence": "Stale down-share was 47.1% vs 51.2% for recent pages; only 22/174 stale pages met the visibility floor.",
        "human_action": "HUMAN_REFRESH_REVIEW",
        "guardrail": "Demote age as a standalone cue; require editor evidence of an actual content issue.",
    },
]
for action in ranked_action_mapping:
    print(action)
print("Saved queue has 10,521 candidates; at 15 minutes each this implies about 2,630.2 assumed review hours.")
print("A 50-page pilot implies 12.5 assumed hours; these are capacity calculations, not measured labor costs.")

## 7. Artifacts the paper embeds

The single local paper is `work/paper/index.html`. Saved notebook outputs report the model comparison and playbook totals, but this workspace does not contain `work/outputs/action_playbook_metrics.json`, `work/outputs/content_action_queue.csv`, or `work/figures/action_queue_by_reason.png`. Reconcile and archive the aggregate metrics and figure before publication; keep the generated queue CSV out of git. Leave `submission/paper_url.txt` as a placeholder until you choose to deploy.

In [ ]:
from pathlib import Path

repo_roots = [Path.cwd(), *Path.cwd().parents]

def locate_repo_item(relative_path):
    return next(
        (root / relative_path for root in repo_roots if (root / relative_path).exists()),
        None,
    )

paper_path = locate_repo_item("work/paper/index.html")
metrics_path = locate_repo_item("work/outputs/action_playbook_metrics.json")
queue_path = locate_repo_item("work/outputs/content_action_queue.csv")
figure_path = locate_repo_item("work/figures/action_queue_by_reason.png")
url_path = locate_repo_item("submission/paper_url.txt")

print(f"Local paper draft: {'present' if paper_path else 'missing'}")
print(f"Aggregate metrics receipt: {'present' if metrics_path else 'not copied to this workspace'}")
print(f"Generated queue CSV: {'present' if queue_path else 'not copied to this workspace'}")
print(f"Reusable action chart: {'present' if figure_path else 'not copied to this workspace'}")
if url_path is not None:
    paper_url_text = url_path.read_text(encoding="utf-8").strip()
    print(f"Deployment URL file is still a placeholder: {paper_url_text.startswith('PASTE-')}")

five_minute_demo = [
    "0:00-0:30 — State the editorial decision and research question.",
    "0:30-1:15 — Describe the anonymized starter snapshot and eligibility filters.",
    "1:15-2:00 — Explain the transparent baseline and reason codes.",
    "2:00-3:15 — Compare the rule and Logistic Regression on the client-held-out split.",
    "3:15-4:15 — Show the leakage challenge, error pattern, and human-review guardrails.",
    "4:15-5:00 — State the limitations and next validation step.",
]
print("Single-paper presentation outline:")
for item in five_minute_demo:
    print(f"- {item}")
print("No reviewer/employer/SEO alternate paper versions are generated.")

## Self-check

This is one research paper, not three audience-specific versions. Keep execution, artifact, and deployment checks honest:

- [x] One conventional research-paper manuscript is present in `work/paper/index.html`
- [x] The manuscript contains the required nine sections, five-sentence abstract, and FlyRank data credit
- [x] Results are drawn from retained notebook outputs, with limitations and provenance caveats stated
- [x] Ranked recommendations reflect the signal audit and remain human-reviewed
- [ ] Reconcile saved notebook outputs and archive aggregate metrics/figure in this workspace
- [ ] Review the public-safety pass and all local/repository links before publication
- [ ] Deploy only after reconciliation; keep `submission/paper_url.txt` unchanged until a real URL exists
- [ ] Review the 5-minute presentation outline